# Ponteiros

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Pegar um endereço com &, segui-lo com *, e usar parâmetros ponteiro para mudar as variáveis de quem chamou ou devolver vários resultados.
- Percorrer um array ou uma string com um ponteiro, e explicar por que a[i] é *(a + i).
- Passar uma função para outra função por um ponteiro para função, como o qsort faz.

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. Toda variável tem um endereço
2. Um ponteiro guarda um endereço
3. Mudando as variáveis de quem chamou
4. Ponteiros e arrays
5. NULL: apontando para lugar nenhum
6. const e ponteiros
7. Erros comuns

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check_int e check_str imprimem ✓ ou ✗; os exercícios abaixo usam
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado %ld, veio %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado \"%s\", veio \"%s\"\n", what, expected, actual);
    }
}

## 1. Toda variável tem um endereço

A memória é uma fileira enorme de bytes numerados, e cada variável fica num **endereço**. `&x` devolve o endereço de `x`; o `printf` mostra endereços com `%p`. Os números mudam de uma execução para outra, mas a ideia não.

In [ ]:
int x = 5;
int y = 7;
printf("x = %d mora em %p\n", x, (void *) &x);
printf("y = %d mora em %p\n", y, (void *) &y);
printf("um int ocupa %zu bytes\n", sizeof x);

🧪 **Experimente:** rode duas vezes: os endereços mudam? Qual a distância entre `x` e `y`?

## 2. Um ponteiro guarda um endereço

Um **ponteiro** é uma variável que guarda um endereço. `int *p = &x;` faz `p` apontar para `x`. Aí `*p` quer dizer "a variável para onde `p` aponta": ler `*p` lê `x`, e `*p = 9;` escreve em `x`. O `*` tem dois trabalhos: numa declaração ele diz "ponteiro"; numa expressão ele **segue** o ponteiro. Os slides mostram isso passo a passo.

#### 🎞️ Um ponteiro, passo a passo

![int x = 5 mora em 0x1000; p = &x guarda esse endereço; *p = 9 escreve 9 em x pelo ponteiro](pointer-steps.pt-br.svg)

In [ ]:
int x = 5;
int *p = &x;
printf("p guarda %p, o endereço de x (%p)\n", (void *) p, (void *) &x);
printf("*p = %d\n", *p);
*p = 9;
printf("depois de *p = 9, x = %d\n", x);
int y = 1;
p = &y;          // agora p aponta para outro lugar
*p += 10;
printf("x = %d, y = %d\n", x, y);

🧪 **Experimente:** preveja a última linha antes de rodar. Depois crie `int *q = p;`: mude `*q` e imprima `y`.

## 3. Mudando as variáveis de quem chamou

Argumentos são cópias, então uma função não consegue mudar o `int` de quem chamou. Mas se receber o **endereço** da variável, ela pode segui-lo com `*` e mudar a própria variável. É por isso que o `scanf` recebe `&qty`. Uma função também pode "devolver" vários resultados assim, por parâmetros ponteiros.

In [ ]:
#include <stdio.h>

void split_time(int total, int *minutes, int *seconds) {
    *minutes = total / 60;
    *seconds = total % 60;
}

int main(void) {
    int m;
    int s;
    split_time(135, &m, &s);
    printf("135 s = %d min %d s\n", m, s);
    return 0;
}

🧪 **Experimente:** chame `split_time(135, m, s)` sem o `&` e leia o aviso: um int não é um endereço.

### ✏️ Exercício: Troque dois ints

Troque os valores de duas variáveis. Uma função em C só recebe cópias, então ela recebe os endereços.

In [ ]:
void swap(int *a, int *b) {
    // TODO: use uma temporária e *a, *b
}

int main(void) {
    int x = 1;
    int y = 2;
    swap(&x, &y);
    check_int("x agora é 2", x, 2);
    check_int("y agora é 1", y, 1);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
void swap(int *a, int *b) {
    int tmp = *a;
    *a = *b;
    *b = tmp;
}

int main(void) {
    int x = 1;
    int y = 2;
    swap(&x, &y);
    check_int("x agora é 2", x, 2);
    check_int("y agora é 1", y, 1);
    return 0;
}
```

</details>

## 4. Ponteiros e arrays

O nome de um array, usado numa expressão, é o **endereço do primeiro elemento**: `a` é `&a[0]`. Somar a um ponteiro o move em **elementos** inteiros, não em bytes: `a + 1` aponta para `a[1]`. É isso que `a[i]` significa de verdade: `*(a + i)`. Um ponteiro também pode percorrer um array com `p++`.

In [ ]:
int a[] = {10, 20, 30, 40};
printf("a == &a[0]? %d\n", a == &a[0]);
printf("*(a + 2) = %d, a[2] = %d\n", *(a + 2), a[2]);
int sum = 0;
for (int *p = a; p < a + 4; p++) {
    sum += *p;
}
printf("soma = %d\n", sum);

🧪 **Experimente:** imprima `(char *) (a + 1) - (char *) a`: um passo de um ponteiro de `int` são quantos bytes?

## 5. NULL: apontando para lugar nenhum

`NULL` é um ponteiro que não aponta para lugar nenhum. Funções o devolvem para dizer "não achei" ou "sem memória". **Sempre confira** se é `NULL` antes de seguir um ponteiro com `*`: seguir o `NULL` derruba o programa.

In [ ]:
const char *text = "ponteiro";
char *hit = strchr(text, 't');
if (hit != NULL) {
    printf("achei '%c' no índice %ld\n", *hit, (long) (hit - text));
}
char *miss = strchr(text, 'z');
if (miss == NULL) {
    printf("não há 'z' em \"%s\"\n", text);
}

🧪 **Experimente:** imprima `*miss` sem a conferência e rode: a verificação de memória acusa o acesso ao endereço 0.

### ✏️ Exercício: Ache um valor

Devolva um ponteiro para o primeiro elemento igual a `target`, ou NULL quando ele não estiver lá.

In [ ]:
int *find(int *a, int n, int target) {
    // TODO: devolva &a[i] quando achar
    return NULL;
}

int main(void) {
    int v[] = {5, 7, 9, 7};
    int *hit = find(v, 4, 7);
    check_int("acha o primeiro 7", hit == &v[1], 1);
    check_int("ausente dá NULL", find(v, 4, 4) == NULL, 1);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
int *find(int *a, int n, int target) {
    for (int i = 0; i < n; i++) {
        if (a[i] == target) {
            return &a[i];
        }
    }
    return NULL;
}

int main(void) {
    int v[] = {5, 7, 9, 7};
    int *hit = find(v, 4, 7);
    check_int("acha o primeiro 7", hit == &v[1], 1);
    check_int("ausente dá NULL", find(v, 4, 4) == NULL, 1);
    return 0;
}
```

</details>

## 6. const e ponteiros

`const int *p` é um ponteiro pelo qual você só pode **ler**: `*p = 3` não compila. Use-o em parâmetros para prometer que a função não vai mudar os dados, como faz o `strlen(const char *s)`. O ponteiro em si ainda pode andar (`p++`).

In [ ]:
#include <stdio.h>

int count_negatives(const int *values, int n) {
    int count = 0;
    for (const int *p = values; p < values + n; p++) {
        if (*p < 0) {
            count++;
        }
    }
    return count;
}

int main(void) {
    int temps[] = {3, -2, 0, -7, 5};
    printf("%d negativos\n", count_negatives(temps, 5));
    return 0;
}

🧪 **Experimente:** acrescente `*p = 0;` dentro do laço e leia o erro.

## 7. Erros comuns

- **Um ponteiro que nunca recebeu valor:** `int *p; *p = 3;` escreve num lugar qualquer. Comece os ponteiros com um endereço real ou com `NULL`.
- **Seguir o `NULL`** sem conferir.
- **Devolver o endereço de uma variável local:** ela some quando a função termina, então o ponteiro fica apontando para nada (o compilador avisa).
- **Confundir `p` com `*p`:** `p = 5` muda para onde ele aponta; `*p = 5` muda o valor que está lá.

In [ ]:
int value = 4;
int *p = NULL;
if (p == NULL) {
    p = &value;
}
*p *= 10;
printf("value = %d\n", value);

🧪 **Experimente:** mude `*p *= 10;` para `p *= 10;` e leia o erro: não dá para multiplicar um endereço.

## Em resumo

Um **ponteiro** guarda o endereço de uma variável. `&x` é o endereço de `x`; se `p` o guarda, `*p` é o próprio `x`, para ler ou mudar. É assim que uma função muda as variáveis de quem a chamou, e como o `scanf` preenche as suas.

```c
#include <stdio.h>

void min_max(const int *values, int n, int *min, int *max) {
    *min = *max = values[0];
    for (const int *p = values + 1; p < values + n; p++) {
        if (*p < *min) *min = *p;
        if (*p > *max) *max = *p;
    }
}

int main(void) {
    int temps[] = {21, 17, 25, 19};
    int low, high;
    min_max(temps, 4, &low, &high);
    printf("%d to %d\n", low, high);    // 17 to 25
    return 0;
}
```

O nome de um array é o endereço do primeiro elemento, então `values + 1` aponta para o elemento seguinte e `a[i]` quer dizer `*(a + i)`.

**Cuidado:** um ponteiro precisa apontar para um lugar válido antes de você usar `*`. Seguir `NULL`, ou um endereço que nunca foi definido, trava ou corrompe a memória.

## Leituras

- [Pointers: Cower In Fear!](https://beej.us/guide/bgc/html/split/pointers.html) (Beej's Guide to C Programming, chapter 5, 12 min): Memória e variáveis, desreferência, passar ponteiros como argumentos (o swap que você não conseguia escrever em Funções), e o ponteiro NULL. A aritmética de ponteiros está no capítulo 11 do mesmo guia.
- [Lecture 4: Pointers](https://cs50.harvard.edu/x/2026/notes/4/#pointers) (CS50x (Harvard), lecture notes, 15 min): Opcional: de "Pointers" até "Pointer Arithmetic", e depois "Swapping". As figuras de endereços e setas deixam a ideia concreta; deixe "Copying and malloc" para a próxima unidade.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Ordenar três** · Fácil
- **Estatísticas por ponteiros** · Fácil
- **Inverter palavras** · Médio
- **Achar todas as ocorrências** · Médio
- **Ordenação por várias chaves** · Difícil